<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_WEEK03_1_Text_Analytics_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **7. 문서 군집화 소개와 실습**

< 문서 군집화 >
- 비슷한 텍스트 구성의 문서를 군집화하는 것
- 학습 데이터 세트가 필요없는 비지도 학습 기반
  - 텍스트 분류 기반의 문서 분류는 사전에 결정 카테고리 값을 가진 학습 데이터 세트가 필요

## Opinion Review 데이터 세트를 이용한 문서 군집화 수행하기

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from google.colab import drive
drive.mount('/content/drive')

# 필수 NLTK 데이터 다운로드
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

def LemNormalize(text):
  lemmatizer = WordNetLemmatizer()

  return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [2]:
import pandas as pd
import glob, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

path = r'/content/drive/MyDrive/ESAA/topics'
all_files = glob.glob(os.path.join(path, "*.data"))
filename_list = []
opinion_text = []

for file_ in all_files:
  df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
  filename_ = file_.split('\\')[-1]
  filename = filename_.split('.')[0]
  filename_list.append(filename)
  opinion_text.append(df.to_string())

document_df = pd.DataFrame({'filename':filename_list, 'opinion_text':opinion_text})
document_df.head()

,filename,opinion_text
0,/content/drive/MyDrive/ESAA/topics/quality_toy...,...
1,/content/drive/MyDrive/ESAA/topics/performance...,...
2,/content/drive/MyDrive/ESAA/topics/price_holid...,...
3,/content/drive/MyDrive/ESAA/topics/mileage_hon...,...
4,/content/drive/MyDrive/ESAA/topics/price_amazo...,...


In [3]:
tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english',
                             ngram_range=(1, 2), min_df=0.05, max_df=0.85)
# opinion_text 칼럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


In [4]:
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_

In [5]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/ESAA/topics/quality_toy...,...,3
1,/content/drive/MyDrive/ESAA/topics/performance...,...,3
2,/content/drive/MyDrive/ESAA/topics/price_holid...,...,2
3,/content/drive/MyDrive/ESAA/topics/mileage_hon...,...,3
4,/content/drive/MyDrive/ESAA/topics/price_amazo...,...,2


In [6]:
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
39,/content/drive/MyDrive/ESAA/topics/accuracy_ga...,...,0
27,/content/drive/MyDrive/ESAA/topics/directions_...,...,0
13,/content/drive/MyDrive/ESAA/topics/display_gar...,...,0
34,/content/drive/MyDrive/ESAA/topics/satellite_g...,...,0
50,/content/drive/MyDrive/ESAA/topics/speed_garmi...,...,0
12,/content/drive/MyDrive/ESAA/topics/updates_gar...,...,0
9,/content/drive/MyDrive/ESAA/topics/voice_garmi...,...,0


In [7]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
37,/content/drive/MyDrive/ESAA/topics/battery-lif...,...,1
18,/content/drive/MyDrive/ESAA/topics/battery-lif...,...,1
41,/content/drive/MyDrive/ESAA/topics/battery-lif...,...,1
21,/content/drive/MyDrive/ESAA/topics/buttons_ama...,...,1
30,/content/drive/MyDrive/ESAA/topics/eyesight-is...,...,1
45,/content/drive/MyDrive/ESAA/topics/features_wi...,...,1
48,/content/drive/MyDrive/ESAA/topics/fonts_amazo...,...,1
11,/content/drive/MyDrive/ESAA/topics/keyboard_ne...,...,1
22,/content/drive/MyDrive/ESAA/topics/navigation_...,...,1
26,/content/drive/MyDrive/ESAA/topics/performance...,...,1


In [8]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
19,/content/drive/MyDrive/ESAA/topics/bathroom_be...,...,2
36,/content/drive/MyDrive/ESAA/topics/food_holida...,...,2
14,/content/drive/MyDrive/ESAA/topics/food_swisso...,...,2
10,/content/drive/MyDrive/ESAA/topics/free_bestwe...,...,2
38,/content/drive/MyDrive/ESAA/topics/location_be...,...,2
33,/content/drive/MyDrive/ESAA/topics/location_ho...,...,2
42,/content/drive/MyDrive/ESAA/topics/parking_bes...,...,2
4,/content/drive/MyDrive/ESAA/topics/price_amazo...,...,2
2,/content/drive/MyDrive/ESAA/topics/price_holid...,...,2
32,/content/drive/MyDrive/ESAA/topics/room_holida...,...,2


In [9]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
16,/content/drive/MyDrive/ESAA/topics/comfort_hon...,...,3
15,/content/drive/MyDrive/ESAA/topics/comfort_toy...,...,3
23,/content/drive/MyDrive/ESAA/topics/gas_mileage...,...,3
49,/content/drive/MyDrive/ESAA/topics/interior_ho...,...,3
25,/content/drive/MyDrive/ESAA/topics/interior_to...,...,3
3,/content/drive/MyDrive/ESAA/topics/mileage_hon...,...,3
1,/content/drive/MyDrive/ESAA/topics/performance...,...,3
0,/content/drive/MyDrive/ESAA/topics/quality_toy...,...,3
47,/content/drive/MyDrive/ESAA/topics/seats_honda...,...,3
6,/content/drive/MyDrive/ESAA/topics/transmissio...,...,3


In [10]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
8,/content/drive/MyDrive/ESAA/topics/sound_ipod_...,headphone jack i got a clear case for it an...,4
29,/content/drive/MyDrive/ESAA/topics/video_ipod_...,...,4


In [11]:
# 3개의 집합으로 군집화
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

# 소속 클러스터를 cluster_label 컬럼으로 할당하고 cluster_label 값으로 정렬
document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
3,/content/drive/MyDrive/ESAA/topics/mileage_hon...,...,0
13,/content/drive/MyDrive/ESAA/topics/display_gar...,...,0
12,/content/drive/MyDrive/ESAA/topics/updates_gar...,...,0
9,/content/drive/MyDrive/ESAA/topics/voice_garmi...,...,0
27,/content/drive/MyDrive/ESAA/topics/directions_...,...,0
23,/content/drive/MyDrive/ESAA/topics/gas_mileage...,...,0
39,/content/drive/MyDrive/ESAA/topics/accuracy_ga...,...,0
34,/content/drive/MyDrive/ESAA/topics/satellite_g...,...,0
50,/content/drive/MyDrive/ESAA/topics/speed_garmi...,...,0
22,/content/drive/MyDrive/ESAA/topics/navigation_...,...,1


In [12]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape ：', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape ： (3, 6155)
[[0.         0.0019285  0.         ... 0.         0.         0.        ]
 [0.00226561 0.00070167 0.         ... 0.         0.         0.        ]
 [0.         0.00027304 0.00060264 ... 0.00117027 0.00092702 0.00092702]]


In [13]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함.
def get_cluster_details(cluster_model, cluster_data, feature_names, cluster_num, top_n_features=10):
  cluster_details = {}

  # cluster_centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환
  # 군집 중심점별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함.
  centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]

  # 개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력
  for cluster_num in range(cluster_num):
    # 개별 군집별 정보를 담을 데이터 초기화.
    cluster_details[cluster_num]= {}
    cluster_details[cluster_num]['cluster'] = cluster_num

    # cluster_centers_.argsort()[:,::-1]로 구한 인덱스를 이용해 top n 피처 단어를 구함.
    top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
    top_features = [feature_names[ind] for ind in top_feature_indexes]

    # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값 구함.
    top_feature_values = cluster_model.cluster_centers_[cluster_num,top_feature_indexes].tolist()

    # cluster_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명 입력
    cluster_details[cluster_num]['top_features'] = top_features
    cluster_details[cluster_num]['top_features_value'] = top_feature_values
    filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']
    filenames = filenames.values.tolist()

    cluster_details[cluster_num]['filenames'] = filenames

  return cluster_details

In [14]:
def print_cluster_details(cluster_details):
  for cluster_num, cluster_detail in cluster_details.items():
    print('###### Cluster {0}'.format(cluster_num))
    print('Top features:', cluster_detail['top_features'])
    print('Reviews 파일명:', cluster_detail['filenames'][:7])
    print('==============================================')

In [15]:
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df, feature_names=feature_names, cluster_num=3, top_n_features=10)

print_cluster_details(cluster_details)

###### Cluster 0
Top features: ['mileage', 'direction', 'map', 'voice', 'satellite', 'accurate', 'speed limit', 'gas mileage', 'gas', 'speed']
Reviews 파일명: ['/content/drive/MyDrive/ESAA/topics/mileage_honda_accord_2008', '/content/drive/MyDrive/ESAA/topics/voice_garmin_nuvi_255W_gps', '/content/drive/MyDrive/ESAA/topics/updates_garmin_nuvi_255W_gps', '/content/drive/MyDrive/ESAA/topics/display_garmin_nuvi_255W_gps', '/content/drive/MyDrive/ESAA/topics/gas_mileage_toyota_camry_2007', '/content/drive/MyDrive/ESAA/topics/directions_garmin_nuvi_255W_gps', '/content/drive/MyDrive/ESAA/topics/satellite_garmin_nuvi_255W_gps']
###### Cluster 1
Top features: ['screen', 'battery', 'battery life', 'keyboard', 'life', 'video', 'size', 'button', 'page', 'feature']
Reviews 파일명: ['/content/drive/MyDrive/ESAA/topics/transmission_toyota_camry_2007', '/content/drive/MyDrive/ESAA/topics/sound_ipod_nano_8gb', '/content/drive/MyDrive/ESAA/topics/keyboard_netbook_1005ha', '/content/drive/MyDrive/ESAA/topics